# Recursive Descent Parser

Objective: To implement a Recursive Descent Parser in Python that accepts grammar productions as input, parses input strings, displays detailed parsing steps, and determines whether the strings are accepted or rejected.

Import Libraries

In [ ]:
from tabulate import tabulate

Define the Recursive Descent Parser

In [ ]:
class RecursiveDescentParser:
    def __init__(self, grammar, start_symbol):
        self.grammar = grammar
        self.start_symbol = start_symbol
        self.steps = []
        self.step_no = 0
        self.input_string = ""
        self.index = 0

    def log(self, step_type, entity, index, description):
        self.steps.append([
            self.step_no, step_type, entity,
            index if index is not None else "",
            description
        ])
        self.step_no += 1

    def parse_non_terminal(self, symbol):
        start_index = self.index
        self.log(
            "Call", symbol, self.index,
            f"Entering parse_non_terminal('{symbol}') at index {self.index}."
        )

        for alt_no, production in enumerate(self.grammar[symbol], start=1):
            self.index = start_index
            self.log(
                "Attempt", symbol, self.index,
                f"Trying alternative {alt_no}: {' '.join(production)} for non-terminal '{symbol}'."
            )

            success = True

            for item in production:
                if item == "eps":
                    continue

                if item in self.grammar:
                    if not self.parse_non_terminal(item):
                        success = False
                        break
                else:
                    if self.index < len(self.input_string) and self.input_string[self.index] == item:
                        self.log(
                            "Match", item, self.index,
                            f"Successfully matched terminal '{item}' with input."
                        )
                        self.index += 1
                    else:
                        self.log(
                            "Match", item, self.index,
                            f"Failed to match terminal '{item}'. Expected '{item}'."
                        )
                        success = False
                        break

            if success:
                self.log(
                    "Success", symbol, self.index,
                    f"Alternative {alt_no}: {' '.join(production)} for '{symbol}' succeeded."
                )
                return True

        self.index = start_index
        self.log(
            "Failure", symbol, self.index,
            f"All alternatives for '{symbol}' failed."
        )
        return False

    def parse(self, input_string):
        self.input_string = input_string
        self.index = 0
        self.steps = []
        self.step_no = 0

        accepted = self.parse_non_terminal(self.start_symbol)
        return accepted and self.index == len(self.input_string)

Read Grammar Productions

In [ ]:
print("Recursive Descent Parser")
print("-------------------------")
print("Enter grammar production rules.")
print("Format: S -> a S a | b")
print("Use 'eps' for epsilon.")

grammar = {}

while True:
    rule = input(" > ").strip()

    if not rule:
        break

    if "->" not in rule:
        print("Invalid production format. Use ->.")
        continue

    lhs, rhs = rule.split("->", 1)
    lhs = lhs.strip()

    if not lhs:
        print("Invalid non-terminal.")
        continue

    grammar.setdefault(lhs, [])

    for alternative in rhs.split("|"):
        symbols = alternative.strip().split()
        grammar[lhs].append(symbols if symbols else ["eps"])

if not grammar:
    raise ValueError("No grammar productions entered.")

start_symbol = next(iter(grammar))
print("\nStart symbol:", start_symbol)

Recursive Descent Parser
-------------------------
Enter grammar production rules.
Format: S -> a S a | b
Use 'eps' for epsilon.
 > S -> a S a | b
 > 

Start symbol: S


Display Grammar

In [ ]:
print("Grammar:")

for symbol, productions in grammar.items():
    alternatives = [" ".join(p) for p in productions]
    print(f"{symbol} -> {' | '.join(alternatives)}")

Grammar:
S -> a S a | b


Enter String and Parse

In [ ]:
parser = RecursiveDescentParser(grammar, start_symbol)

input_string = input("Enter the string to parse: ")

print(f"\n--- Parsing '{input_string}' with start symbol '{start_symbol}' ---")

accepted = parser.parse(input_string)

print("\n--- Detailed Parsing Steps ---")
print(tabulate(
    parser.steps,
    headers=["Step", "Type", "Entity", "Index", "Description"],
    tablefmt="grid"
))

if accepted:
    print(f"\nResult: String '{input_string}' is ACCEPTED by the grammar.")
else:
    print(f"\nResult: String '{input_string}' is REJECTED by the grammar.")

Enter the string to parse: aba

--- Parsing 'aba' with start symbol 'S' ---

--- Detailed Parsing Steps ---
+--------+---------+----------+---------+---------------------------------------------------+
|   Step | Type    | Entity   |   Index | Description                                       |
+========+=========+==========+=========+===================================================+
|      0 | Call    | S        |       0 | Entering parse_non_terminal('S') at index 0.      |
+--------+---------+----------+---------+---------------------------------------------------+
|      1 | Attempt | S        |       0 | Trying alternative 1: a S a for non-terminal 'S'. |
+--------+---------+----------+---------+---------------------------------------------------+
|      2 | Match   | a        |       0 | Successfully matched terminal 'a' with input.     |
+--------+---------+----------+---------+---------------------------------------------------+
|      3 | Call    | S        |       1 | Ente

Test Valid Strings

In [ ]:
valid_strings = ["b", "aba", "aabaa", "aaabaaa"]

for string in valid_strings:
    accepted = parser.parse(string)
    print(f"Input: {string} -> {'ACCEPTED' if accepted and parser.index == len(string) else 'REJECTED'}")

Input: b -> ACCEPTED
Input: aba -> ACCEPTED
Input: aabaa -> ACCEPTED
Input: aaabaaa -> ACCEPTED


Test Invalid Strings

In [ ]:
invalid_strings = ["ab", "abb", "aab", "abc", ""]

for string in invalid_strings:
    accepted = parser.parse(string)
    print(f"Input: '{string}' -> {'ACCEPTED' if accepted and parser.index == len(string) else 'REJECTED'}")

Input: 'ab' -> REJECTED
Input: 'abb' -> REJECTED
Input: 'aab' -> REJECTED
Input: 'abc' -> REJECTED
Input: '' -> REJECTED


# Algorithm

1. Read the grammar productions from the user.
2. Store each non-terminal and its alternatives in a dictionary.
3. Select the first non-terminal as the start symbol.
4. Read the input string and initialize the input index to zero.
5. Call the recursive parsing function for the start symbol.
6. Try each production alternative, resetting the index when an alternative fails.
7. Match terminals with the current input character.
8. Recursively parse non-terminals.
9. Treat `eps` as an empty production.
10. Record each call, attempt, match, success, and failure.
11. Accept the string only if parsing succeeds and the entire input is consumed.
12. Otherwise, reject the string.